# Design and source verification — W2048409281, attempt 01

## Retained scope and original-source checks

This attempt covers exactly the six pooled OLS fits in Table 4: Panel A at t+1 and Panel B at t+2, each for fiscal years 1974–1983, 1984–1993, and 1994–2005. The paper defines Eq. (1) on physical pp. 14–15 and describes its population on physical p. 6. There are no Table 3 logit, Table 5 payout-rank, repurchase, or later regular-payer fits in this attempt.

The source requires U.S.-incorporated NYSE/AMEX/NASDAQ firms, excluding utilities and financial firms, and defines adjusted earnings as IB (#18) + 0.6×interest expense (#15) − 0.6×interest income (#62). Both current E(t) and forward E(t+h) use total assets from t−1 as the common denominator, including the two-year horizon. Table 4's note identifies dividends as item #21 and states that accounting rates are winsorized at ±100%. The paper's Table 1 note on physical p. 9 calls out annual dividends and earnings #18/#21, but provides no separate missing-interest-item rule or numeric industry-code boundaries.

The modern annual item mapping was checked against the cached provider guide `output/reproduction_runs/reference_documents/compustat_xpressfeed_understanding_2007.pdf`: physical p. 18 (printed p. 12) explicitly maps Xpressfeed `IDIT` to Interest Income; physical p. 21 (printed p. 15) maps `NOPIO` to nonoperating income excluding interest income; physical p. 80 (printed p. 74) lists IDIT as potentially combined with SALE, XINT, NOPI, or NOPIO. The exact local catalog maps `COMP.FUNDA.IDIT` as “Interest and Related Income - Total”; it has no literal IINT or IDIT_DC/NOPI_DC/NOPIO_DC columns. The local `TII` field is labeled “Interest Income - Total (Financial Services)”; the 2004 provider guide maps TII to annual item A321, a financial-services item that includes interest and dividends from earning assets. TII is not used for the source's item #62.

The construction uses observed IDIT directly. It does not set missing interest income to zero and does not use NOPI−NOPIO as a substitute. In a bounded current-year population audit, NOPI and NOPIO were jointly observed on 127,735 of 128,290 rows; IDIT was observed on 81,152 rows, and on those overlapping rows IDIT matched NOPI−NOPIO to floating-point precision. This supports the direct IDIT mapping but does not authorize filling rows with missing IDIT. The guide's combined-item table and absent local data-code fields also mean that rowwise combined status cannot be independently audited in this local delivery. Missing IDIT therefore leaves earnings unavailable; the resulting complete-case loss is documented against the paper's reported Ns rather than repaired by imputation.

## Source rows, temporal alignment, and screens

The annual source is catalog-listed `COMP.FUNDA`, using INDL, consolidated (C), domestic population (D), standard-format (STD) rows for fiscal years 1973–2007. This horizon includes the t−1 asset rows for the earliest estimation year and the one-/two-year leads through FY2007. The observed source dates run from 30Jun1973 through 31May2008. No current-year classification is imposed on lagged or future accounting rows.

The filtered source has two duplicate GVKEY/FYEAR keys, four rows for GVKEY 006557 across FY1982–1983. All four have missing IDIT. Only unique source-year keys feed current-year and future-earnings rows, so neither duplicated fiscal year can supply an earnings numerator. For lagged assets, the build now uses a field-specific support table: it admits a GVKEY/FYEAR only when exactly one source row has nonmissing AT and that row's own-date USD conversion is positive. Each duplicate key has one such row (DATADATE 31Aug, CURCD CAD) and one fully blank statement row; this retains the uniquely observed total-assets input without choosing between observed values or joining duplicate statements. The two rows with observed IDIT remain missing, and no earnings are imputed. The affected firm has no observed IDIT in any 1974–2005 current-year record, so restoring these lag-asset values has no effect on any Table 4 regression sample or estimate. Native build diagnostics report the duplicate-key and unique-asset counts.

The source asks for U.S. incorporation but the authorized catalog has no dated incorporation-history table. The current-year screen uses row-level `FUNDA.FIC='USA'`; FIC is labeled “Current ISO Country Code - Incorporation,” so this remains a timing limitation. Current headers, headquarters location, and native currency are not used as historical incorporation evidence.

For exchange history, `CRSP.CCMXPF_LINKTABLE` is matched to `CRSP.MSENAMES` at the current annual DATADATE. Links require `LINKPRIM` P/C and `LINKTYPE` LC/LU, with missing start/end dates treated as open bounds. A qualifying date-valid row must have common-share code 10/11 and exchange code 1/2/3 (NYSE/AMEX/NASDAQ). This is a firm-level “at least one linked common share” interpretation of the paper's listing rule, rather than a current-header exchange screen.

Industry uses valid `FUNDA.SICH` (100–9999) as the primary historical SIC. If SICH is missing or zero, it uses SICCD from one unique dated CRSP name code linked through CCM at DATADATE. Multiple distinct fallback SICCD codes or no code leave classification unresolved. When both valid SICH and CRSP SICCD are present, SICH controls and disagreements are counted. SICH is unavailable/zero for essentially all screened source rows in 1974–1981; the dated CRSP fallback preserves that early period. From 1982–1986 SICH remains sparse, then increases sharply in 1987. In the overlap audit, exact four-digit SICs agreed on 24.8% of 99,448 rows, while the financial/utility inclusion screen agreed on 96.1% using financial SIC 6000–6999 and utility SIC 4900–4999. The paper excludes utilities and financial firms but supplies no SIC cutoffs; these conventional code ranges are a disclosed best-effort rule, with valid SICH always primary.

The bounded pre-estimation screen audit finds 128,290 current-year firm-years from 1974–2005 before unique-period and accounting-input completeness. Of these, 81,152 have IDIT; NOPI/NOPIO coverage and the IDIT comparison are diagnostics only. The six final regression Ns will be taken from the native model samples and compared with, not forced to, the paper's counts.

## Currency and earnings-rate construction

IB, XINT, IDIT, and DVC are annual flows translated using each row's own CURCD and DATADATE. The Compustat Xpressfeed currency guide (cached 2011 guide, physical pp. 86–90; printed pp. 76–82) documents the GBP-intermediary cross-rate method. For non-USD records the build multiplies the reported amount by EXRAT12M(GBP→USD) / EXRAT12M(GBP→CURCD); USD is the identity. Total assets use daily EXRATD legs at their own period-end date with the analogous GBP bridge. CURNCD is not the source reporting currency, and EXRATTPD is not used as a currency-eligibility screen. Missing/nonpositive rate legs leave the converted amount unavailable.

For each source row, adjusted earnings in USD are (IB + 0.6×XINT − 0.6×IDIT)×flow rate. An observed positive DVC sets DP=1 and an observed zero sets DP=0; missing or negative DVC remains unavailable. Loss firms remain eligible. The current rate is E(t)/AT(t−1), and the horizon outcomes are E(t+1)/AT(t−1) or E(t+2)/AT(t−1). Assets must be positive. All ratios are fixed-capped to [−1,1], not percentile-winsorized. No future dividends or future-year country, exchange, or industry screens are required.

Table 1's aggregate earnings are unadjusted item #18 and its population/aggregate unit is not identical to the interest-adjusted Table 4 model variable. The Table 1 total is not used as the Table 4 outcome.

## Estimation and inference

Each subperiod fit is OLS with an intercept, DP, current adjusted earnings rate, and their interaction. No fixed effects are reported or added. The installed `reghdfe` 6.12.3 supports two-way clustering without absorption; an isolated syntax probe confirmed `noabsorb vce(cluster firm year)` runs and retains the constant. The model cells use that direct command with separate firm and fiscal-year clusters; no package was installed.

The frozen Stata template exports coefficient estimates and p-values in parentheses (`cells(b(star fmt(3)) p(par))`) across MD, RTF, CSV, and TeX. The paper reports standard errors in parentheses, so the native `reghdfe` standard errors below are compared with the paper's SEs; exported p-values are not interpreted as SEs. Each fit also has descriptive statistics for its actual complete-case observations.

| Fit | N (paper / reproduced) | DP × E (paper / reproduced) | SE (paper / reproduced) | p (reproduced) | Adjusted R² (paper / reproduced) |
|---|---:|---:|---:|---:|---:|
| Panel A, h=1, 1974–1983 | 23,042 / 16,378 | .0314 / .0772 | .011 / .0496 | .154 | .63 / .5155 |
| Panel A, h=1, 1984–1993 | 30,938 / 23,208 | .0795 / −.0490 | .0084 / .0709 | .507 | .70 / .5070 |
| Panel A, h=1, 1994–2005 | 42,619 / 27,813 | .0643 / −.0910 | .0068 / .0459 | .073 | .70 / .5714 |
| Panel B, h=2, 1974–1983 | 18,950 / 14,647 | .0245 / .1436 | .019 / .0851 | .126 | .43 / .2849 |
| Panel B, h=2, 1984–1993 | 27,769 / 20,661 | .0985 / .0038 | .012 / .0787 | .963 | .53 / .3414 |
| Panel B, h=2, 1994–2005 | 38,579 / 24,343 | .0992 / −.1116 | .010 / .0890 | .236 | .54 / .3866 |

The reproduced estimates do not match the paper's checks. The lower Ns are consistent with requiring observed modern IDIT after the provider mapping, but the missing-item rule in the paper is unspecified; this does not account by itself for the coefficient, SE, and fit differences. The outputs are retained as a transparent best-effort replication, with no tuning to the published values. The MD/RTF/CSV/TeX regression exports preserve the template's coefficient-and-p-value layout, and the six descriptive tables summarize the actual regression samples.

The predecessor's initial SAS build failed because it requested the literal IINT field; its failure and corrected IDIT build are preserved in `../attempt_01/runtime/`. A2 adds a field-specific lag-asset support rule for the two duplicate GVKEY/FYEAR keys, while retaining the provider-mapped IDIT complete-case rule for earnings. The final A2 native SAS build is `runtime/20261103T064545819994Z/` (return code 0); a fresh Stata run completed all six fits and six sample-descriptive cells in `runtime/20261103T064558563417Z/` (all return code 0). SAS reported 334,857 raw rows, 334,853 unique earnings periods, 334,855 lag-asset support keys including two duplicate keys with unique AT, and no duplicate monthly or daily FX keys. The predecessor's bounded diagnostics under `../attempt_01/runtime/diagnostics/item62/` were also reviewed for IDIT coverage, SIC fallback/agreement, and duplicate periods.


In [ ]:
options nodate nonumber errorabend;
libname cp "/storage/sas/base/comp/d_na" access=readonly;
libname fx "/storage/sas/base/comp/d_global/currency" access=readonly;
libname ccm "/storage/sas/base/crsp/a_ccm" access=readonly;
libname crsp "/storage/sas/base/crsp/a_stock" access=readonly;
libname out "./data";

title "Verify annual source fields used in the modern item-62 mapping";
proc sql;
  select name, type, length, label
  from dictionary.columns
  where libname='CP' and memname='FUNDA'
    and upcase(name) in ('IB','XINT','IDIT','IINT','IDIT_DC','NOPI_DC',
                         'NOPIO_DC','AT','DVC','FIC','SICH','CURCD',
                         'FYEAR','DATADATE')
  order by name;
quit;

title "Annual Compustat source coverage: fiscal years 1973-2007";
proc sql;
  create table out.raw_annual as
  select gvkey, datadate, fyear, indfmt, consol, popsrc, datafmt,
         curcd, fic, sich, at, ib, xint, idit, dvc
  from cp.funda
  where indfmt='INDL' and consol='C' and popsrc='D' and datafmt='STD'
    and fyear>=1973 and fyear<=2007
  order by gvkey, fyear, datadate;
  select count(*) as raw_rows,
         count(distinct gvkey) as distinct_firms,
         min(fyear) as first_fyear,
         max(fyear) as last_fyear,
         min(datadate) format=date9. as first_actual_date,
         max(datadate) format=date9. as last_actual_date
  from out.raw_annual;
  create table out.fy_counts as
  select gvkey, fyear, count(*) as n_source_rows
  from out.raw_annual
  group by gvkey, fyear;
  create table out.dup_fy as
  select gvkey, fyear, n_source_rows as n_rows
  from out.fy_counts
  where n_source_rows>1;
  select count(*) as duplicate_firm_year_groups,
         coalesce(sum(n_rows-1),0) as excess_rows,
         coalesce(sum(n_rows),0) as ambig_fy_source_rows
  from out.dup_fy;
quit;

title "Raw annual field and FX coverage by fiscal year";
data out.fx_mth;
  set fx.g_exrt_mth(keep=datadate fromcurm tocurm exrat12m);
  where fromcurm='GBP' and datadate>='01JAN1973'd and datadate<='31DEC2008'd;
run;
data out.fx_dly;
  set fx.g_exrt_dly(keep=datadate fromcurd tocurd exratd exrattpd);
  where fromcurd='GBP' and datadate>='01JAN1973'd and datadate<='31DEC2008'd;
run;
proc sql;
  create table out.dup_fx_mth as
  select datadate, tocurm, count(*) as n_rows
  from out.fx_mth
  group by datadate, tocurm
  having calculated n_rows>1;
  create table out.dup_fx_dly as
  select datadate, tocurd, count(*) as n_rows
  from out.fx_dly
  group by datadate, tocurd
  having calculated n_rows>1;
  select count(*) as duplicate_monthly_fx_keys from out.dup_fx_mth;
  select count(*) as duplicate_daily_fx_keys from out.dup_fx_dly;
  create table out.annual_rates as
  select a.*,
         case
           when upcase(a.curcd)='USD' then 1
           when fm.exrat12m>0 and fu.exrat12m>0
             then fu.exrat12m/fm.exrat12m
           else .
         end as flow_to_usd,
         case
           when upcase(a.curcd)='USD' then 1
           when ds.exratd>0 and du.exratd>0
             then du.exratd/ds.exratd
           else .
         end as stock_to_usd,
         (a.ib+0.6*a.xint-0.6*a.idit) *
           (case
              when upcase(a.curcd)='USD' then 1
              when fm.exrat12m>0 and fu.exrat12m>0
                then fu.exrat12m/fm.exrat12m
              else .
            end) as earn_usd,
         a.at *
           (case
              when upcase(a.curcd)='USD' then 1
              when ds.exratd>0 and du.exratd>0
                then du.exratd/ds.exratd
              else .
            end) as at_usd
  from out.raw_annual as a
  left join out.fx_mth as fm
    on a.datadate=fm.datadate and upcase(a.curcd)=upcase(fm.tocurm)
  left join out.fx_mth as fu
    on a.datadate=fu.datadate and upcase(fu.tocurm)='USD'
  left join out.fx_dly as ds
    on a.datadate=ds.datadate and upcase(a.curcd)=upcase(ds.tocurd)
  left join out.fx_dly as du
    on a.datadate=du.datadate and upcase(du.tocurd)='USD';
  select fyear,
         count(*) as raw_rows,
         sum(not missing(ib)) as n_ib,
         sum(not missing(xint)) as n_xint,
         sum(not missing(idit)) as n_idit,
         sum(missing(idit)) as n_idit_missing,
         sum(not missing(dvc)) as n_dvc,
         sum(upcase(curcd)='USD') as n_reported_usd,
         sum(missing(curcd)) as n_missing_curcd,
         sum(not missing(flow_to_usd)) as n_flow_fx,
         sum(missing(flow_to_usd)) as n_flow_fx_missing,
         sum(not missing(earn_usd)) as n_adjusted_earnings_usd,
         sum(not missing(at_usd) and at_usd>0) as n_positive_assets_usd,
         min(datadate) format=date9. as first_actual_date,
         max(datadate) format=date9. as last_actual_date
  from out.annual_rates
  group by fyear
  order by fyear;
  create table out.fic_by_firm as
  select gvkey, count(distinct fic) as n_fic_values
  from out.raw_annual
  where not missing(fic)
  group by gvkey;
  select count(*) as firms_with_fic,
         sum(n_fic_values>1) as firms_with_row_fic_changes
  from out.fic_by_firm;
  create table out.annual_unique as
  select a.*
  from out.annual_rates as a
  inner join out.fy_counts as c
    on a.gvkey=c.gvkey and a.fyear=c.fyear
  where c.n_source_rows=1
  order by a.gvkey, a.fyear, a.datadate;
  select count(*) as unique_annual_period_rows,
         count(distinct gvkey) as firms_with_unique_annual_periods,
         min(fyear) as first_unique_fyear,
         max(fyear) as last_unique_fyear
  from out.annual_unique;
  /* Earnings rows remain restricted to unique GVKEY/FYEAR keys. For the
     role-specific lagged-asset input, retain a duplicate key only when the
     source contains exactly one nonmissing AT and it converts to positive
     USD at that row's own DATADATE. This recovers a unique field value without
     choosing among competing annual statements. */
  create table out.asset_key_counts as
  select gvkey, fyear,
         count(*) as n_source_rows,
         sum(not missing(at)) as n_observed_at,
         sum(not missing(at) and at>0 and not missing(at_usd) and at_usd>0)
           as n_positive_usd_at
  from out.annual_rates
  group by gvkey, fyear;
  create table out.lag_asset_support as
  select a.gvkey, a.fyear, a.datadate, a.curcd, a.at,
         a.stock_to_usd, a.at_usd
  from out.annual_rates as a
  inner join out.asset_key_counts as c
    on a.gvkey=c.gvkey and a.fyear=c.fyear
  where c.n_observed_at=1 and c.n_positive_usd_at=1
    and not missing(a.at) and a.at>0
    and not missing(a.at_usd) and a.at_usd>0;
  select count(*) as lag_asset_support_keys,
         sum(n_source_rows>1 and n_observed_at=1 and n_positive_usd_at=1)
           as dup_keys_unique_lag_at
  from out.asset_key_counts;
  select p.gvkey, p.fyear, p.datadate format=date9. as asset_date,
         p.curcd as asset_currency, p.at as source_at,
         p.stock_to_usd, p.at_usd as converted_assets_usd,
         c.n_source_rows
  from out.lag_asset_support as p
  inner join out.asset_key_counts as c
    on p.gvkey=c.gvkey and p.fyear=c.fyear
  where c.n_source_rows>1
  order by p.gvkey, p.fyear;
quit;

title "Date-valid historical CCM and CRSP name records for current-year screens";
data out.cur_rows;
  set out.annual_unique;
  if fyear>=1974 and fyear<=2005;
run;
proc sql;
  create table out.stock_rows as
  select c.gvkey, c.fyear, c.datadate,
         l.lpermno as permno_link, l.linkprim, l.linktype,
         n.permno, n.namedt, n.nameendt, n.exchcd, n.shrcd, n.siccd
  from out.cur_rows as c
  inner join ccm.ccmxpf_linktable as l
    on c.gvkey=l.gvkey
   and l.linkprim in ('P','C')
   and l.linktype in ('LC','LU')
   and not missing(l.lpermno)
   and (missing(l.linkdt) or l.linkdt<=c.datadate)
   and (missing(l.linkenddt) or l.linkenddt>=c.datadate)
  inner join crsp.msenames as n
    on l.lpermno=n.permno
   and not missing(n.namedt)
   and n.namedt<=c.datadate
   and (missing(n.nameendt) or n.nameendt>=c.datadate);
  create table out.stock_agg as
  select gvkey, fyear, datadate,
         count(distinct permno_link) as n_linked_permnos,
         count(*) as n_historical_name_rows,
         max(case when exchcd in (1,2,3) and shrcd in (10,11)
                  then 1 else 0 end) as exchange_ok,
         sum(case when exchcd in (1,2,3) and shrcd in (10,11)
                  then 1 else 0 end) as n_common_market_rows,
         count(distinct case
           when siccd>=100 and siccd<=9999 then siccd else .
         end) as n_valid_siccd,
         min(case when siccd>=100 and siccd<=9999 then siccd else . end)
           as min_siccd,
         max(case when siccd>=100 and siccd<=9999 then siccd else . end)
           as max_siccd
  from out.stock_rows
  group by gvkey, fyear, datadate;
quit;
proc sort data=out.cur_rows;
  by gvkey fyear datadate;
run;
proc sort data=out.stock_agg;
  by gvkey fyear datadate;
run;
data out.current_info;
  merge out.cur_rows(in=in_cur) out.stock_agg;
  by gvkey fyear datadate;
  if in_cur;
  if missing(exchange_ok) then exchange_ok=0;
  if missing(n_linked_permnos) then n_linked_permnos=0;
  if missing(n_historical_name_rows) then n_historical_name_rows=0;
  sich_valid=(not missing(sich) and sich>=100 and sich<=9999);
  if sich_valid then do;
    sic_used=sich;
    sic_source=1;
  end;
  else if n_valid_siccd=1 then do;
    sic_used=min_siccd;
    sic_source=2;
  end;
  else do;
    sic_used=.;
    sic_source=0;
  end;
  sic_disagree=(sich_valid and n_valid_siccd>0 and (min_siccd ne sich or max_siccd ne sich));
  if not missing(sic_used) then do;
    if (sic_used>=4900 and sic_used<=4999) or
       (sic_used>=6000 and sic_used<=6999) then industry_ok=0;
    else industry_ok=1;
  end;
  else industry_ok=0;
  country_ok=(upcase(strip(fic))='USA');
  exchange_ok=(exchange_ok=1);
run;
proc sort data=out.current_info;
  by gvkey fyear datadate;
run;
title "Annual historical classification and exchange coverage at current fiscal dates";
proc sql;
  select fyear,
         count(*) as current_year_rows,
         sum(country_ok) as fic_usa,
         sum(missing(fic)) as fic_missing,
         sum(sich_valid) as valid_historical_sich,
         sum(n_valid_siccd=1) as unambiguous_dated_crsp_siccd,
         sum(sic_source=2) as crsp_sic_fallback_rows,
         sum(sic_source=0) as unresolved_industry_rows,
         sum(sic_disagree) as sich_siccd_disagreements,
         sum(n_valid_siccd>1) as multiple_crsp_siccd_rows,
         sum(exchange_ok) as common_nyse_amex_nasdaq_rows,
         sum(n_linked_permnos=0) as no_dated_ccm_name_match
  from out.current_info
  group by fyear
  order by fyear;
  select count(*) as current_screen_rows,
         sum(country_ok) as fic_usa,
         sum(exchange_ok) as historical_exchange_pass,
         sum(industry_ok) as n_industry_ok,
         sum(sic_source=1) as sourced_from_funda_sich,
         sum(sic_source=2) as sourced_from_crsp_siccd_fallback,
         sum(sic_disagree) as overlapping_code_disagreements,
         sum(n_valid_siccd>1 and not sich_valid) as ambiguous_fallback_rows,
         sum(not missing(dvc)) as observed_common_dividends,
         sum(not missing(idit)) as observed_idit,
         sum(missing(idit)) as missing_idit,
         sum(not missing(earn_usd)) as n_earn_adj
  from out.current_info;
quit;

title "Construct lagged assets and forward earnings without future-year classification screens";
proc sql;
  create table out.model_rows as
  select c.gvkey, c.fyear, c.datadate,
         c.fic, c.curcd, c.sich, c.sic_used, c.sic_source,
         c.sic_disagree, c.n_valid_siccd, c.n_linked_permnos,
         c.country_ok, c.exchange_ok, c.industry_ok,
         c.dvc, c.ib, c.xint, c.idit,
         c.flow_to_usd, c.stock_to_usd, c.earn_usd,
         p.datadate as lag_datadate,
         p.at_usd as assets_lag_usd,
         h1.datadate as lead1_datadate,
         h1.earn_usd as earnings_h1_usd,
         h2.datadate as lead2_datadate,
         h2.earn_usd as earnings_h2_usd
  from out.current_info as c
  left join out.lag_asset_support as p
    on c.gvkey=p.gvkey and p.fyear=c.fyear-1
   and p.datadate<c.datadate
  left join out.annual_unique as h1
    on c.gvkey=h1.gvkey and h1.fyear=c.fyear+1
   and h1.datadate>c.datadate
  left join out.annual_unique as h2
    on c.gvkey=h2.gvkey and h2.fyear=c.fyear+2
   and h2.datadate>c.datadate
  order by c.gvkey, c.fyear, c.datadate;
quit;
data out.main;
  set out.model_rows;
  if dvc>0 then dp=1;
  else if dvc=0 then dp=0;
  else dp=.;
  current_ok=(country_ok=1 and exchange_ok=1 and industry_ok=1
              and not missing(dp) and not missing(earn_usd));
  rate_ok=(current_ok=1 and not missing(assets_lag_usd)
           and assets_lag_usd>0);
  if rate_ok then e_rate=max(-1,min(1,earn_usd/assets_lag_usd));
  else e_rate=.;
  use_h1=(rate_ok=1 and not missing(earnings_h1_usd));
  use_h2=(rate_ok=1 and not missing(earnings_h2_usd));
  if use_h1 then y_h1=max(-1,min(1,earnings_h1_usd/assets_lag_usd));
  else y_h1=.;
  if use_h2 then y_h2=max(-1,min(1,earnings_h2_usd/assets_lag_usd));
  else y_h2=.;
  label gvkey="Compustat global company key"
        fyear="Compustat fiscal year"
        datadate="Actual fiscal period end"
        dvc="Cash dividends common, Compustat item 21"
        dp="Regular common dividend indicator"
        e_rate="Current adjusted earnings / lagged total assets"
        y_h1="One-year-ahead adjusted earnings / lagged total assets"
        y_h2="Two-year-ahead adjusted earnings / lagged total assets"
        assets_lag_usd="Total assets at fiscal year t-1, USD"
        sic_used="Historical SIC used for current-year screen"
        sic_source="1=FUNDA SICH; 2=dated CRSP SICCD fallback"
        use_h1="Eligible Table 4 Panel A fit row"
        use_h2="Eligible Table 4 Panel B fit row";
run;
title "Current-year field availability and estimation-sample counts";
proc sql;
  select fyear,
         count(*) as current_year_rows,
         sum(current_ok) as current_source_screen_pass,
         sum(rate_ok) as current_and_lag_asset_pass,
         sum(missing(idit)) as missing_current_idit,
         sum(use_h1) as h1_fit_rows,
         sum(use_h2) as h2_fit_rows
  from out.main
  group by fyear
  order by fyear;
  select 'Panel A h=1 1974-1983' as fit length=32, count(*) as N
    from out.main where use_h1=1 and fyear>=1974 and fyear<=1983
  union all
  select 'Panel A h=1 1984-1993', count(*)
    from out.main where use_h1=1 and fyear>=1984 and fyear<=1993
  union all
  select 'Panel A h=1 1994-2005', count(*)
    from out.main where use_h1=1 and fyear>=1994 and fyear<=2005
  union all
  select 'Panel B h=2 1974-1983', count(*)
    from out.main where use_h2=1 and fyear>=1974 and fyear<=1983
  union all
  select 'Panel B h=2 1984-1993', count(*)
    from out.main where use_h2=1 and fyear>=1984 and fyear<=1993
  union all
  select 'Panel B h=2 1994-2005', count(*)
    from out.main where use_h2=1 and fyear>=1994 and fyear<=2005;
quit;
data out.main;
  set out.main;
  keep gvkey fyear datadate fic curcd dvc dp e_rate y_h1 y_h2
       use_h1 use_h2 current_ok rate_ok assets_lag_usd
       sic_used sic_source sic_disagree n_valid_siccd n_linked_permnos
       flow_to_usd stock_to_usd;
run;
proc datasets library=out nolist;
  delete raw_annual dup_fy fy_counts fx_mth fx_dly dup_fx_mth dup_fx_dly
         annual_rates annual_unique asset_key_counts lag_asset_support
         fic_by_firm cur_rows stock_rows
         stock_agg current_info model_rows;
quit;
title;
